In [ ]:
import pandapower as pp
import pp_heig_plot as pp_plot
import pp_heig_simulation as pp_sim
from pp_importer import PandaPowerImporter
import pandas as pd
from pandapower.plotting.plotly import simple_plotly

net_file_path = "data/all_data.xlsx"

net = pp.create.create_empty_network()
net = PandaPowerImporter().read_excel(file_path=net_file_path)

print("Nombre de bus :", len(net.bus))
print("Nombre de lignes :", len(net.line))
print("Nombre de transformateurs :", len(net.trafo))
print("Nombre de charges :", len(net.load))

In [ ]:
import json
from pandapower.plotting.plotly import simple_plotly

# Positions approximatives tirées de la carte simplifiée
positions = {
    0: (10.5, 6.5),  # Réseau MT
    1: (3.0, 11.0),  # CDBT012139
    2: (7.5, 2.0),  # CDBT004774
    3: (7.5, 4.5),  # CDBT901452
    4: (10.0, 6.5),  # STMT003438
    5: (1.0, 4.5),  # N1
    6: (6.0, -2.0),  # CDBT901604
    7: (5.5, 10.0),  # CDBT004760
    8: (8.0, 8.5),  # CDBT004764
    9: (7.8, 11.5),  # CDBT003746
    10: (1.0, 12.0),  # CDBT900784
    11: (8.5, -0.5),  # 60437
}

# Position du bus CDBT016055 (Bus 13)
positions[12] = (5.5, 0.5)

# Position du réseau MT, proche du transformateur
positions[0] = (10.8, 6.5)

# Position du poste STMT003438
positions[4] = (10.0, 6.5)

# Attribuer les coordonnées
for bus, (x, y) in positions.items():
    net.bus.at[bus, "geo"] = json.dumps({"type": "Point", "coordinates": [x, y]})

# Afficher le réseau
fig = simple_plotly(net, auto_open=False, use_line_geo=False, on_map=False)

fig.show()

In [ ]:
# test réseaux charge

# Importation des profils
profils = pd.read_excel("data/3_bus_power_profile.xlsx", sheet_name="load", header=None)


# Vérifier les charges importées
display(net.load)

# Lancer le Load Flow
pp.runpp(net)

print("Convergence :", net.converged)

# Résultats des tensions
display(net.res_bus)

# Résultats des lignes
display(net.res_line)

# Résultats du transformateur
display(net.res_trafo)

pp.runpp(net)
pp_plot.plot_powerflow_result(
    net=net, plot_title="3-bus powerflow results"  # , filename="3_bus_pp_result"
)
net.res_bus

In [ ]:
# charges

charge = batiments["NumÈro EGID"].dropna().astype(str).str.strip()
charge = charge[charge.ne("")].unique()

# Associe chaque nom de cabinet à l'index du bus correspondant dans net
bus_par_cabinet = (
    net.bus.dropna(subset=["name"]).reset_index().set_index("name")["index"].to_dict()
)

for _, ligne in batiments.iterrows():
    egid = ligne["NumÈro EGID"]
    cabinet = str(ligne["Cabinet"]).strip()

    if pd.isna(egid) or cabinet not in bus_par_cabinet:
        continue

    bus = bus_par_cabinet[cabinet]

    pp.create_load(
        net,
        bus=bus,
        p_mw=0.1,
        q_mvar=0.05,
        name=str(egid),
    )

In [ ]:
# lignes

cables = [
    ("CDBT900784", "CDBT012139", 0.090, "GKT 3x50/50"),
    ("CDBT012139", "CDBT004760", 0.080, "GKT 3x50/50"),
    ("CDBT004760", "CDBT004764", 0.115, "GKN 3x95/95"),
    ("CDBT003746", "CDBT004764", 0.100, "GKN 3x150/150"),
    ("CDBT004764", "STMT003438", 0.180, "GKN 3x150/150"),
    ("STMT003438", "CDBT901452", 0.135, "GKN 3x150/150"),
    ("N1", "CDBT901452", 0.255, "GKN 3x50/50"),
    ("CDBT901452", "CDBT004774", 0.100, "GKN 3x95/95"),
    ("CDBT004774", "CDBT016055", 0.085, "GKN 3x50/50"),
    ("CDBT004774", "CDBT901604", 0.205, "GKN 3x150/150"),
    ("STMT003438", "60437", 0.340, "GKN 3x240/240"),
]

parametres_cables = {
    "GKT 3x50/50": (0.642, 0.083, 210, 0.142),
    "GKN 3x50/50": (0.641, 0.082, 205, 0.142),
    "GKN 3x95/95": (0.320, 0.080, 225, 0.220),
    "GKN 3x150/150": (0.206, 0.078, 245, 0.290),
    "GKN 3x240/240": (0.125, 0.075, 275, 0.415),
}


bus_par_nom = {str(nom).strip(): idx for idx, nom in net.bus["name"].items()}

for depart, arrivee, longueur_km, type_cable in cables:
    depart = str(depart).strip()
    arrivee = str(arrivee).strip()

    if depart not in bus_par_nom:
        raise ValueError(f"Bus de départ absent de net.bus : {depart}")

    if arrivee not in bus_par_nom:
        raise ValueError(f"Bus d'arrivée absent de net.bus : {arrivee}")

    if type_cable not in parametres_cables:
        raise ValueError(f"Paramètres inconnus pour le câble : {type_cable}")

    r, x, c, max_i = parametres_cables[type_cable]

    pp.create_line_from_parameters(
        net,
        from_bus=bus_par_nom[depart],
        to_bus=bus_par_nom[arrivee],
        length_km=longueur_km,
        r_ohm_per_km=r,
        x_ohm_per_km=x,
        c_nf_per_km=c,
        max_i_ka=max_i,
        name=f"{depart} - {arrivee} ({type_cable})",
    )

print(
    net.line[
        [
            "name",
            "from_bus",
            "to_bus",
            "length_km",
            "r_ohm_per_km",
            "x_ohm_per_km",
            "c_nf_per_km",
            "max_i_ka",
        ]
    ]
)

In [ ]:
# transfo de quartier

pp.create_transformer_from_parameters(
    net,
    hv_bus=,
    lv_bus=,
    sn_mva=0.4,
    vk_percent=6,
    vkr_percent=1.425,
    i0_percent=0.3375,
    pfe_kw=1.35,
    vn_hv_kv=20.0,
    vn_lv_kv=0.4,
    name="trafo1",
)

In [ ]:
# Noeud bilan

ext_grid = pp.create_ext_grid(net, bus=bus_transfo, vm_pu=1.02, name="slack")

In [ ]:
from pandapower.plotting.plotly import simple_plotly

fig = simple_plotly(net, auto_open=False)
fig.show()

In [ ]:
print("Bus :", len(net.bus))
print("Réseau externe :", len(net.ext_grid))
print("Lignes :", len(net.line))
print("Transformateurs :", len(net.trafo))
print(net.ext_grid)